[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-04-response-models.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Response Models and Status Codes
**certified-journeys / fastapi-certified** · Practice · Day 4 of FastAPI for Python Engineers

> **Goal for today:** Control exactly what your API returns — filter fields with `response_model`, return correct HTTP status codes, and raise meaningful errors with `HTTPException`.


In [ ]:
%pip install -q fastapi httpx


## Step 1 · Why Response Models? Field Filtering with `response_model=`

FastAPI lets you define **two separate Pydantic models** for a route:

| Model | Purpose |
|---|---|
| `UserCreate` | Input — accepts `password` |
| `UserOut` | Output — strips `password` before returning |

When you set `response_model=UserOut` on a route, FastAPI:
1. Validates the return value against `UserOut`
2. Serializes **only** the fields declared in `UserOut`
3. Documents the correct shape in the OpenAPI schema

This is the primary guardrail against leaking internal fields like passwords, tokens, or internal IDs.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

# --- Pydantic models ---
class UserCreate(BaseModel):
    username: str
    email: str
    password: str          # sensitive — must never leave the API

class UserOut(BaseModel):
    username: str
    email: str             # password is intentionally absent

# Simulated in-memory "database"
fake_users_db: dict[int, dict] = {}
next_id = 1

# response_model= tells FastAPI to filter the return value through UserOut
@app.post("/users", response_model=UserOut)
def create_user(user: UserCreate):
    global next_id
    record = {"id": next_id, **user.dict()}  # stores password internally
    fake_users_db[next_id] = record
    next_id += 1
    return record  # FastAPI filters this through UserOut before sending

client = TestClient(app)

resp = client.post("/users", json={"username": "alice", "email": "alice@example.com", "password": "s3cret"})
print(resp.status_code)   # 200
print(resp.json())         # {"username": "alice", "email": "alice@example.com"} — no password!


**What just happened?**

- The handler returned a dict that included `password` and `id`.
- **FastAPI serialized only the fields declared in `UserOut`** — `username` and `email`.
- `password` and `id` never touched the wire, even though the dict contained them.
- The OpenAPI docs at `/docs` will now show `UserOut` as the response schema.


## Step 2 · Avoiding Null Noise with `response_model_exclude_unset=True`

When a Pydantic model has **optional fields with defaults**, FastAPI normally sends all of them — including unset ones with their default values. This pollutes responses with fields the client never asked for.

Set `response_model_exclude_unset=True` to send **only fields that were explicitly set**.

| Scenario | Without flag | With flag |
|---|---|---|
| Only `username` returned | `{"username":"alice", "bio": null, "avatar": null}` | `{"username":"alice"}` |


In [ ]:
from typing import Optional

app2 = FastAPI()

class ProfileOut(BaseModel):
    username: str
    bio: Optional[str] = None
    avatar: Optional[str] = None

# Without exclude_unset — all optional fields appear with null
@app2.get("/profile/verbose", response_model=ProfileOut)
def profile_verbose():
    return ProfileOut(username="alice")  # bio and avatar are defaults (None)

# With exclude_unset — only explicitly set fields are sent
@app2.get("/profile/clean", response_model=ProfileOut, response_model_exclude_unset=True)
def profile_clean():
    return ProfileOut(username="alice")  # same object, cleaner wire format

client2 = TestClient(app2)

verbose = client2.get("/profile/verbose").json()
clean   = client2.get("/profile/clean").json()

print("Verbose:", verbose)   # {'username': 'alice', 'bio': None, 'avatar': None}
print("Clean:  ", clean)     # {'username': 'alice'}


**What just happened?**

- Both routes return the **same Python object** — a `ProfileOut` with only `username` set.
- Without `exclude_unset`, Pydantic includes all fields, sending `null` for `bio` and `avatar`.
- **With `exclude_unset=True`, only `username` — the only field explicitly set — is serialized.**
- This is especially valuable for partial-update (PATCH) endpoints where you never want to overwrite fields not included in the request.


## Step 3 · Correct HTTP Status Codes with `status_code=`

HTTP status codes communicate semantics — not just success or failure:

| Operation | Correct code | Meaning |
|---|---|---|
| GET, PUT | 200 | OK — resource returned |
| POST (create) | 201 | Created — new resource made |
| DELETE | 204 | No Content — done, nothing to return |
| Not found | 404 | Resource does not exist |
| Conflict | 409 | Duplicate or constraint violation |

FastAPI defaults every route to `200`. Set `status_code=` to override per route.


In [ ]:
from fastapi import FastAPI, status

app3 = FastAPI()

items_db: dict[int, str] = {1: "Hammer", 2: "Wrench"}
item_counter = 3

# POST → 201 Created
@app3.post("/items", status_code=status.HTTP_201_CREATED)
def create_item(name: str):
    global item_counter
    items_db[item_counter] = name
    item_counter += 1
    return {"id": item_counter - 1, "name": name}

# GET → 200 OK (default, but explicit for clarity)
@app3.get("/items/{item_id}", status_code=status.HTTP_200_OK)
def get_item(item_id: int):
    return {"id": item_id, "name": items_db.get(item_id, "unknown")}

# DELETE → 204 No Content (no body should be returned)
@app3.delete("/items/{item_id}", status_code=status.HTTP_204_NO_CONTENT)
def delete_item(item_id: int):
    items_db.pop(item_id, None)
    # Return None — 204 means "no body"

client3 = TestClient(app3)

r_post   = client3.post("/items", params={"name": "Screwdriver"})
r_get    = client3.get("/items/1")
r_delete = client3.delete("/items/2")

print("POST  status:", r_post.status_code)    # 201
print("GET   status:", r_get.status_code)     # 200
print("DELETE status:", r_delete.status_code) # 204


**What just happened?**

- **`status.HTTP_201_CREATED`** is the semantic constant for 201 — prefer constants over magic numbers.
- **204 No Content** expects no body. FastAPI correctly suppresses the body when `None` is returned.
- Using the `fastapi.status` module gives you autocomplete and avoids typos.
- These status codes appear correctly in the OpenAPI docs and are tested by clients that enforce them.


## Step 4 · Raising `HTTPException` for Error Responses

When a resource is missing or a request is invalid, raise `HTTPException` instead of returning a 200 with an error body. This is idiomatic FastAPI and ensures HTTP clients behave correctly.

```python
from fastapi import HTTPException
raise HTTPException(status_code=404, detail="Item not found")
```

The `detail` field is JSON-serialized and placed in the response body under `{"detail": "..."}`. You can pass a string, a dict, or any JSON-serializable value.


In [ ]:
from fastapi import FastAPI, HTTPException

app4 = FastAPI()

products: dict[int, dict] = {
    1: {"name": "Widget", "price": 9.99},
    2: {"name": "Gadget", "price": 24.99},
}

@app4.get("/products/{product_id}")
def get_product(product_id: int):
    if product_id not in products:
        # FastAPI catches this and returns a 404 JSON response
        raise HTTPException(
            status_code=404,
            detail=f"Product {product_id} not found"
        )
    return products[product_id]

@app4.get("/products")
def list_products(min_price: float = 0.0):
    if min_price < 0:
        raise HTTPException(
            status_code=422,
            detail={"msg": "min_price must be >= 0", "given": min_price}
        )
    return [p for p in products.values() if p["price"] >= min_price]

client4 = TestClient(app4)

# Happy path
ok = client4.get("/products/1")
print("Found:    ", ok.status_code, ok.json())

# Missing product
missing = client4.get("/products/99")
print("Missing:  ", missing.status_code, missing.json())

# Invalid query param — dict detail
bad = client4.get("/products", params={"min_price": -5})
print("Bad param:", bad.status_code, bad.json())


**What just happened?**

- `HTTPException` is caught by FastAPI's built-in exception handler and serialized to `{"detail": ...}`.
- The `detail` argument accepts **any JSON-serializable value** — strings, dicts, lists.
- HTTP clients (browsers, `curl`, SDKs) correctly detect the error and don't treat it as a success.
- **Never return `{"error": "not found"}` with a 200 status** — this breaks HTTP semantics and confuses monitoring tools.


## Step 5 · Custom Exception Handlers with `@app.exception_handler()`

For **domain-specific exceptions** (e.g., `InsufficientStockError`, `AuthorizationError`), define a custom exception class and register a handler. This keeps business logic exceptions decoupled from HTTP concerns.

```python
@app.exception_handler(MyDomainError)
async def my_error_handler(request, exc):
    return JSONResponse(status_code=400, content={"msg": str(exc)})
```

The handler receives the `Request` and the exception instance. Return a `JSONResponse` (or any `Response` subclass) with the appropriate status code.


In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

# --- Domain exceptions (pure Python, no FastAPI import needed here) ---
class InsufficientStockError(Exception):
    def __init__(self, product_id: int, requested: int, available: int):
        self.product_id = product_id
        self.requested = requested
        self.available = available

app5 = FastAPI()

# Register the handler — FastAPI calls this when InsufficientStockError is raised
@app5.exception_handler(InsufficientStockError)
async def insufficient_stock_handler(request: Request, exc: InsufficientStockError):
    return JSONResponse(
        status_code=409,   # 409 Conflict — request conflicts with current state
        content={
            "error": "insufficient_stock",
            "product_id": exc.product_id,
            "requested": exc.requested,
            "available": exc.available,
        }
    )

inventory = {1: 5, 2: 0}  # product_id → quantity

@app5.post("/order")
def place_order(product_id: int, quantity: int):
    stock = inventory.get(product_id, 0)
    if quantity > stock:
        # Raise the domain exception — the handler above converts it to HTTP
        raise InsufficientStockError(product_id, quantity, stock)
    inventory[product_id] -= quantity
    return {"message": "Order placed", "remaining": inventory[product_id]}

client5 = TestClient(app5)

# Sufficient stock
ok = client5.post("/order", params={"product_id": 1, "quantity": 3})
print("OK:", ok.status_code, ok.json())

# Out of stock
err = client5.post("/order", params={"product_id": 2, "quantity": 1})
print("Conflict:", err.status_code, err.json())


**What just happened?**

- `InsufficientStockError` is a **plain Python exception** — it knows nothing about HTTP.
- The `@app5.exception_handler()` decorator bridges the gap, converting domain exceptions to HTTP responses.
- **The route handler raises `InsufficientStockError` cleanly** — no `JSONResponse` imports or HTTP logic inside business code.
- This pattern scales well: add handlers for `AuthorizationError`, `RateLimitError`, etc. without changing route code.


## Step 6 · Combining Response Models and Status Codes on Real Routes

In production APIs you combine all three tools on every route:
- `response_model=` → field filtering + OpenAPI docs
- `status_code=` → semantic HTTP code for the happy path
- `HTTPException` → semantic HTTP code for error paths

The pattern below models a minimal but complete user registration endpoint.


In [ ]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, EmailStr
from typing import Optional

# EmailStr requires: pip install pydantic[email]
# We'll use str here to keep Colab dependency-free
class UserIn(BaseModel):
    username: str
    email: str
    password: str

class UserOut(BaseModel):
    id: int
    username: str
    email: str
    # password is absent

app6 = FastAPI()

users: dict[int, dict] = {}
emails: set[str] = set()  # enforce uniqueness
uid = 1

@app6.post(
    "/users",
    response_model=UserOut,          # filters password from output
    status_code=status.HTTP_201_CREATED,  # signals resource created
)
def register(user: UserIn):
    global uid
    if user.email in emails:
        raise HTTPException(
            status_code=status.HTTP_409_CONFLICT,
            detail=f"Email {user.email!r} is already registered"
        )
    record = {"id": uid, "username": user.username, "email": user.email, "password": user.password}
    users[uid] = record
    emails.add(user.email)
    uid += 1
    return record  # response_model strips password

@app6.get("/users/{user_id}", response_model=UserOut)
def get_user(user_id: int):
    if user_id not in users:
        raise HTTPException(status_code=404, detail="User not found")
    return users[user_id]

client6 = TestClient(app6)

# Register
r1 = client6.post("/users", json={"username": "bob", "email": "bob@example.com", "password": "hunter2"})
print("Register:", r1.status_code, r1.json())  # 201, no password

# Duplicate
r2 = client6.post("/users", json={"username": "bob2", "email": "bob@example.com", "password": "other"})
print("Duplicate:", r2.status_code, r2.json())  # 409

# Get existing
r3 = client6.get("/users/1")
print("Get user:", r3.status_code, r3.json())  # 200, no password

# Get missing
r4 = client6.get("/users/99")
print("Missing:", r4.status_code, r4.json())  # 404


**What just happened?**

- All three tools work together on the same route: `response_model`, `status_code`, and `HTTPException`.
- **201 for creation, 409 for conflict, 404 for missing** — each path returns the semantically correct code.
- The `password` field is completely absent from both the 201 and 200 responses.
- This pattern is production-ready — copy it to your own projects as a starting template.


## Step 7 · Inspecting OpenAPI Schema — Validating Your Response Models

FastAPI generates an OpenAPI schema from your routes. You can inspect it programmatically to confirm that `response_model` is working correctly — this is useful for automated testing in CI.


In [ ]:
import json

# Use the last app (app6) — fetch the generated OpenAPI schema
schema_resp = client6.get("/openapi.json")
schema = schema_resp.json()

# Inspect what the POST /users route advertises as its response
post_users = schema["paths"]["/users"]["post"]
response_201 = post_users["responses"]["201"]
print("POST /users response schema ref:")
print(json.dumps(response_201, indent=2))

# Inspect the UserOut model definition in the schema
user_out_schema = schema["components"]["schemas"]["UserOut"]
print("\nUserOut properties:", list(user_out_schema["properties"].keys()))
# Should NOT include 'password'


**What just happened?**

- FastAPI's OpenAPI schema is generated at startup from your route decorators and Pydantic models.
- **`UserOut` properties confirm `password` is absent** — not just at runtime, but in the API contract.
- Clients that generate code from OpenAPI (e.g., `openapi-generator`) will never see `password` in their generated models.
- You can add schema assertions to your test suite to catch accidental field leakage during code review.


## Step 8 · Default Response Headers and Custom Error Bodies

Two advanced tricks worth knowing:
1. **`headers=` on `HTTPException`** — attach custom HTTP headers to error responses (e.g., `WWW-Authenticate` for 401).
2. **`responses=` on the route** — document additional non-default status codes in OpenAPI without changing the handler.


In [ ]:
from fastapi import FastAPI, HTTPException

app7 = FastAPI()

VALID_TOKEN = "secret-token-abc"

# The `responses` dict documents extra codes in OpenAPI without affecting runtime
@app7.get(
    "/secure-data",
    responses={
        401: {"description": "Missing or invalid Authorization token"},
        403: {"description": "Valid token but insufficient permissions"},
    }
)
def secure_data(token: str = ""):
    if not token:
        raise HTTPException(
            status_code=401,
            detail="Authorization token required",
            headers={"WWW-Authenticate": "Bearer"},  # standard 401 header
        )
    if token != VALID_TOKEN:
        raise HTTPException(status_code=403, detail="Forbidden")
    return {"data": "top secret"}

client7 = TestClient(app7, raise_server_exceptions=False)

# No token
r_no_token = client7.get("/secure-data")
print("No token:", r_no_token.status_code)
print("  WWW-Authenticate header:", r_no_token.headers.get("www-authenticate"))

# Wrong token
r_bad = client7.get("/secure-data", params={"token": "wrong"})
print("Bad token:", r_bad.status_code, r_bad.json())

# Correct token
r_ok = client7.get("/secure-data", params={"token": VALID_TOKEN})
print("Good token:", r_ok.status_code, r_ok.json())


**What just happened?**

- **`headers=` on `HTTPException`** lets you attach standard HTTP headers (like `WWW-Authenticate`) to error responses — required by RFC 7235 for 401 responses.
- **`responses=` on the route decorator** adds documentation hints to OpenAPI without affecting the handler logic.
- **401 vs 403**: 401 means "I don't know who you are", 403 means "I know who you are, but you can't do this".
- These details matter when building client SDKs and monitoring dashboards that react to specific status codes.


In [ ]:
# Challenge: Response models and exceptions
#
# Build a /books API with:
#   - BookCreate model: title (str), author (str), secret_notes (str)
#   - BookOut model: id (int), title (str), author (str)  [no secret_notes]
#   - POST /books → 201, response_model=BookOut
#   - GET /books/{id} → 200, response_model=BookOut; raise 404 if missing
#   - DELETE /books/{id} → 204; raise 404 if missing
#   - A custom exception OutOfPrintError and its handler (returns 410 Gone)
#
# Verify with TestClient:
#   1. POST returns 201 with no secret_notes in body
#   2. GET returns 200 for existing, 404 for missing
#   3. DELETE returns 204 for existing, 404 for missing

from fastapi import FastAPI, HTTPException, status, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel

# Your solution here
# class OutOfPrintError(Exception): ...
# class BookCreate(BaseModel): ...
# class BookOut(BaseModel): ...
# app = FastAPI()
# @app.exception_handler(OutOfPrintError) ...
# @app.post("/books", ...) ...
# ...
# client = TestClient(app)
# assert client.post(...).status_code == 201
# assert "secret_notes" not in client.post(...).json()


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `response_model=` | Filters output fields through a Pydantic model; prevents internal field leakage |
| `response_model_exclude_unset=True` | Only serializes explicitly-set fields; ideal for PATCH responses |
| `status_code=` | Sets the happy-path HTTP code; use `fastapi.status` constants |
| `HTTPException` | Raises HTTP errors with a JSON body; `detail` is any JSON-serializable value |
| `@app.exception_handler()` | Maps domain exceptions to HTTP responses; keeps business logic HTTP-free |
| 201 / 204 / 404 / 409 | Created / No Content / Not Found / Conflict — match the semantic |
| `headers=` on HTTPException | Attach standard headers (e.g. `WWW-Authenticate`) to error responses |

> **Tip:** Use `response_model=` to control exactly what leaves your API — never accidentally expose internal fields.

---
## What's next
**Day 5** → Dependency Injection — learn how `Depends()` makes your routes testable, composable, and DRY.

Mark Day 4 complete in your [tracker](../index.html).
